# Limpieza catálogo

Construye y revisa el catálogo aprobado cuando `RUN` sea `True`.

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd

ROOT = Path.cwd()
if (ROOT / 'Suplematch-Backend').exists():
    ROOT = ROOT / 'Suplematch-Backend'
while ROOT.name != 'Suplematch-Backend' and ROOT.parent != ROOT:
    ROOT = ROOT.parent

script = ROOT / 'scripts/build_approved_catalog.py'
assert script.exists()
ROOT

## Insumos

In [ ]:
inputs = {
    'scraped': ROOT / 'data/raw/pharmacies/supplements_exhaustive_clean.csv',
    'digemid': ROOT / 'data/raw/digemid/digemid_limpio.csv',
    'components': ROOT / 'data/training/supplement_model/product_components.csv',
}
pd.DataFrame([{'name': key, 'path': str(path.relative_to(ROOT)), 'exists': path.exists(), 'size': path.stat().st_size if path.exists() else 0} for key, path in inputs.items()])

## Construcción

In [ ]:
RUN = False
command = ['python', str(script)]
result = None
if RUN:
    result = subprocess.run(command, cwd=ROOT, text=True, capture_output=True, check=True)
{'command': ' '.join(command), 'executed': RUN, 'stdout': result.stdout[-2000:] if result else ''}

## Calidad

In [ ]:
catalog = ROOT / 'data/catalog/approved_catalog.csv'
if catalog.exists():
    df = pd.read_csv(catalog, keep_default_na=False)
    quality = {
        'rows': len(df),
        'pharmacies': df['pharmacy'].nunique() if 'pharmacy' in df else 0,
        'with_registro': int((df.get('registro_sanitario', '') != '').sum()) if 'registro_sanitario' in df else 0,
        'with_component_id': int((df.get('component_id', '') != '').sum()) if 'component_id' in df else 0,
    }
    display(quality)
    display(df.head(10))
else:
    {'rows': 0}